In [1]:
import pandas as pd
import numpy as np
import glob

In [2]:
df = pd.read_csv('aggregated_voting_NYCMayor.csv')
df.head()

,1st,2nd,3rd,4th,5th,count
0,217572,Null,Null,Null,Null,64274
1,217796,Null,Null,Null,Null,20566
2,219469,Null,Null,Null,Null,12771
3,219978,Null,Null,Null,Null,10258
4,217572,219469,Null,Null,Null,9833


In [24]:
rank_cols = ['1st', '2nd', '3rd', '4th', '5th']
df_clean = df[df[rank_cols].notna().all(axis = 1)]

def clean_ballot(row):
    ranked = []
    for c in row:
        if c in ["Null", "Overvote", "Undervote", "Write-in"]:
            break
        ranked.append(c)
    return ranked

ballots = [(list(row[:-1]), row[-1]) for row in df_clean.values.tolist()]

candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        candidates.add(candidate)
candidates = list(candidates)

print(f"Number of Ballots: {len(ballots)}")
print(f"Candidates: {candidates}")


            
    

Number of Ballots: 62154
Candidates: ['218491', '221141', 'Write-in', '221183', '218117', '219469', '217654', '219978', '221458', '217572', '218127', '218922', '217796', 'Null', '217605']


In [5]:
df_clean.to_csv('cleaned_aggregated_voting_NYCMayor.csv', index = False)

In [6]:
df_clean.head()

,1st,2nd,3rd,4th,5th,count
0,217572,Null,Null,Null,Null,64274
1,217796,Null,Null,Null,Null,20566
2,219469,Null,Null,Null,Null,12771
3,219978,Null,Null,Null,Null,10258
4,217572,219469,Null,Null,Null,9833


In [7]:
df = pd.read_csv('aggregated_voting_NYCMayor.csv')

In [9]:
first_column = df.columns[0]
number_counts = df[first_column].value_counts()
print(f"number of unique values in first column: {len(number_counts)}")
print(number_counts)

number of unique values in first column: 14
1st
217572      10490
219469       9003
217796       8394
219978       7231
217605       5031
218491       4854
221183       3766
218127       3675
221141       2544
217654       2543
218117       1491
218922       1433
221458       1216
Write-in      483
Name: count, dtype: int64


In [11]:
ballots_df = pd.read_csv('cleaned_aggregated_voting_NYCMayor.csv')

In [27]:
ballots = [(clean_ballot(row[:-1]), row[-1]) for row in df_clean.values.tolist()]
candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        if candidate not in {"Null", "Overvote", "Undervote", "Write-in"}:
            candidates.add(candidate)
candidates = list(candidates)

def count_votes(ballots, candidates):
    """
    Count the votes for the current round using weighted ballots.
    Each ballot contributes its weight to the vote of the highest-ranked candidate
    that hasn't been eliminated.
    """
    vote_counts = {candidate: 0 for candidate in candidates}
    for ballot, weight in ballots:
        for candidate in ballot:
            if candidate in candidates:
                vote_counts[candidate] += weight
                break  # Only count the top remaining candidate.
    return vote_counts

round_num = 1
while True:
    # Tally the weighted votes for this round.
    vote_counts = count_votes(ballots, candidates)
    total_votes = sum(vote_counts.values())
    print(f"Round {round_num} vote counts: {vote_counts} (Total active votes: {total_votes})")

    # Check if any candidate has a majority (>50% of active votes)
    for candidate, count in vote_counts.items():
        if count > total_votes / 2:
            print(f"\nWinner: {candidate} with {count} votes (majority of {total_votes} active votes)!")
            winner = candidate
            break
    else:
        # No candidate has a majority; eliminate candidate(s) with the fewest votes.
        min_votes = min(vote_counts.values())
        eliminated = [candidate for candidate, count in vote_counts.items() if count == min_votes]
        print(f"Eliminated in round {round_num}: {eliminated}\n")
         # Remove the eliminated candidate(s) from further consideration.
        for candidate in eliminated:
            candidates.remove(candidate)
        
        # If no candidates remain or all remaining are tied, declare a tie.
        if not candidates or len(vote_counts) == len(eliminated):
            print("Election resulted in a tie among the remaining candidates:")
            print(vote_counts)
            break

        round_num += 1
        continue
    break

Round 1 vote counts: {'218491': 26741, '221141': 7181, '221183': 23419, '218117': 4022, '219469': 201494, '217654': 7869, '219978': 184826, '221458': 2309, '217572': 289786, '218127': 25477, '218922': 2745, '217796': 115498, '217605': 52068} (Total active votes: 943435)
Eliminated in round 1: ['221458']

Round 2 vote counts: {'218491': 26855, '221141': 7210, '221183': 23549, '218117': 4080, '219469': 201819, '217654': 7932, '219978': 184923, '217572': 290235, '218127': 25624, '218922': 2846, '217796': 115700, '217605': 52169} (Total active votes: 942942)
Eliminated in round 2: ['218922']

Round 3 vote counts: {'218491': 27195, '221141': 7347, '221183': 23663, '218117': 4147, '219469': 202235, '217654': 7999, '219978': 185205, '217572': 290604, '218127': 25742, '217796': 115967, '217605': 52281} (Total active votes: 942385)
Eliminated in round 3: ['218117']

Round 4 vote counts: {'218491': 27665, '221141': 7683, '221183': 23740, '219469': 203949, '217654': 8055, '219978': 185556, '21757

In [25]:
ballots = [(clean_ballot(row[:-1]), row[-1]) for row in df_clean.values.tolist()]
from itertools import combinations
candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        if candidate not in {"Null", "Write-in", "Undervote", "Overvote"}:
            candidates.add(candidate)
candidates = list(candidates)
print(len(candidates), candidates)
def simulate_copeland_fast(ballots, candidates):
    cand_index = {c: i for i, c in enumerate(candidates)}
    n = len(candidates)
    matrix = np.zeros((n, n))
    
    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_index]
        unranked = [c for c in candidates if c not in ranked]
        
        # Ranked candidates vs each other: order on the ballot determines the win
        for c1, c2 in combinations(ranked, 2):
            matrix[cand_index[c1]][cand_index[c2]] += weight
        
        # Every ranked candidate beats every unranked candidate
        for c1 in ranked:
            for c2 in unranked:
                matrix[cand_index[c1]][cand_index[c2]] += weight
        
        # Unranked vs unranked: no preference was expressed, so skip (tie)
    
    scores = {c: 0 for c in candidates}
    for c1, c2 in combinations(candidates, 2):
        i, j = cand_index[c1], cand_index[c2]
        if matrix[i][j] > matrix[j][i]:
            scores[c1] += 1
            scores[c2] -= 1
        elif matrix[j][i] > matrix[i][j]:
            scores[c2] += 1
            scores[c1] -= 1
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\nCopeland Scores:")
    for candidate, score in sorted_scores:
        print(f"  {candidate}: {score}")
    winner = sorted_scores[0][0]
    print(f"\nCopeland Winner: {winner}")
    return winner

13 ['218491', '221141', '221183', '218117', '219469', '217654', '219978', '221458', '217572', '218127', '218922', '217796', '217605']


In [26]:
copeland_winner = simulate_copeland_fast(ballots, candidates)


Copeland Scores:
  217572: 12
  219469: 10
  219978: 8
  217796: 6
  217605: 4
  221183: 2
  218491: 0
  218127: -2
  221141: -4
  218922: -6
  218117: -8
  221458: -10
  217654: -12

Copeland Winner: 217572
